# Polynomial Regression Experiment

## 1. Objective

Evaluate whether introducing polynomial features can capture nonlinear
relationships between the predictors and `Standard_yield` and improve
predictive performance over the OLS baseline.

### Hypothesis

The exploratory analysis suggested that some predictors may have
nonlinear relationships with `Standard_yield`. Polynomial regression
will be evaluated to determine whether explicitly modelling these
relationships improves predictive performance.

### Evaluation Criteria

Models will be evaluated using:

- RMSE — lower is better
- $R^2$ — higher is better

The OLS baseline provides the reference point:

- RMSE: 0.067770
- $R^2$: 0.655339

In [1]:
cd ..

/home/dataflix/Projects/maji-ndogo-yield-intelligence


In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import(
    StandardScaler,
    OneHotEncoder,
    PolynomialFeatures
)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [3]:
from src.config import config_params
from src.field_data_processor import FieldDataProcessor
field_df = FieldDataProcessor(config_params).process()
field_df.head()

2026-08-21 21:21:00 | src.data_ingestion | INFO | Starting data ingestion
2026-08-21 21:21:00 | src.field_data_processor | INFO | FieldDataProcessor is initialized
2026-08-21 21:21:00 | src.data_ingestion | INFO | Successfully connected to sqlite:///data/Maji_Ndogo_farm_survey_small.db
2026-08-21 21:21:00 | src.data_ingestion | INFO | Query executed successfully. Rows: 5654
2026-08-21 21:21:00 | src.field_data_processor | INFO | SQL data is successfully loaded into the pandas DataFrame
2026-08-21 21:21:00 | src.field_data_processor | INFO | Swapped columns: Annual_yield with Crop_type
2026-08-21 21:21:00 | src.field_data_processor | INFO | Converted the negative elavtion values to absulte figures.
2026-08-21 21:21:00 | src.field_data_processor | INFO | Mispelled crop names and extra spaces were found and got fixed
2026-08-21 21:21:00 | src.data_ingestion | INFO | Attempting to read CSV from: https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_m

,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
0,786.05580,-7.389911,-7.556202,Rural_Akatsi,14.795113,1125.2,-3.1,33.1,15.00,0.62,Sandy,6.169393,0.085267,1.3,0.751354,cassava,0.577964
1,674.33410,-7.736849,-1.051539,Rural_Sokoto,11.374611,1450.7,-3.9,30.6,13.35,0.64,Volcanic,5.676648,0.399684,2.2,1.069865,cassava,0.486302
2,826.53390,-9.926616,0.115156,Rural_Sokoto,11.339692,2208.9,-1.8,28.4,13.30,0.69,Volcanic,5.331993,0.358029,3.4,2.208801,tea,0.649647
3,574.94617,-2.420131,-6.592215,Rural_Kilimani,7.109855,328.8,-5.8,32.2,13.20,0.54,Loamy,5.328150,0.286687,2.4,1.277635,cassava,0.532348
4,886.35300,-3.055434,-7.952609,Rural_Kilimani,55.007656,785.2,-2.5,31.0,14.25,0.72,Sandy,5.721234,0.043190,1.5,0.832614,wheat,0.555076


In [4]:
display(field_df.shape)
display(field_df.columns.to_list())
print(field_df.info())


(5654, 17)

['Elevation',
 'Latitude',
 'Longitude',
 'Location',
 'Slope',
 'Rainfall',
 'Min_temperature_C',
 'Max_temperature_C',
 'Ave_temps',
 'Soil_fertility',
 'Soil_type',
 'pH',
 'Pollution_level',
 'Plot_size',
 'Annual_yield',
 'Crop_type',
 'Standard_yield']

<class 'pandas.DataFrame'>
RangeIndex: 5654 entries, 0 to 5653
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Elevation          5654 non-null   float64
 1   Latitude           5654 non-null   float64
 2   Longitude          5654 non-null   float64
 3   Location           5654 non-null   str    
 4   Slope              5654 non-null   float64
 5   Rainfall           5654 non-null   float64
 6   Min_temperature_C  5654 non-null   float64
 7   Max_temperature_C  5654 non-null   float64
 8   Ave_temps          5654 non-null   float64
 9   Soil_fertility     5654 non-null   float64
 10  Soil_type          5654 non-null   str    
 11  pH                 5654 non-null   float64
 12  Pollution_level    5654 non-null   float64
 13  Plot_size          5654 non-null   float64
 14  Annual_yield       5654 non-null   float64
 15  Crop_type          5654 non-null   str    
 16  Standard_yield     5654 non-null   

### X and y split

In [5]:
def X_y_feature_selection(field_df) -> pd.DataFrame | pd.Series:
    features = [col for col in field_df.columns if col not in ["Standard_yield", "Annual_yield"]]
    X = field_df[features]
    y = field_df["Standard_yield"]
    return X, y

X, y = X_y_feature_selection(field_df)
print(X.info())

<class 'pandas.DataFrame'>
RangeIndex: 5654 entries, 0 to 5653
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Elevation          5654 non-null   float64
 1   Latitude           5654 non-null   float64
 2   Longitude          5654 non-null   float64
 3   Location           5654 non-null   str    
 4   Slope              5654 non-null   float64
 5   Rainfall           5654 non-null   float64
 6   Min_temperature_C  5654 non-null   float64
 7   Max_temperature_C  5654 non-null   float64
 8   Ave_temps          5654 non-null   float64
 9   Soil_fertility     5654 non-null   float64
 10  Soil_type          5654 non-null   str    
 11  pH                 5654 non-null   float64
 12  Pollution_level    5654 non-null   float64
 13  Plot_size          5654 non-null   float64
 14  Crop_type          5654 non-null   str    
dtypes: float64(12), str(3)
memory usage: 662.7 KB
None


### Train Test Split

In [6]:
def X_y_train_test_split(X, y)-> pd.DataFrame | pd.Series:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = X_y_train_test_split(X, y)

### Feature data types selection

In [7]:
def numerical_categorical(X_train):
    num_features = X_train.select_dtypes(include = ["number"]).columns.tolist()
    cat_features = X_train.select_dtypes(include = ["str"]).columns.tolist()
    return num_features, cat_features
num_features, cat_features = numerical_categorical(X)
display(num_features)
display(cat_features)
type(num_features)

['Elevation',
 'Latitude',
 'Longitude',
 'Slope',
 'Rainfall',
 'Min_temperature_C',
 'Max_temperature_C',
 'Ave_temps',
 'Soil_fertility',
 'pH',
 'Pollution_level',
 'Plot_size']

['Location', 'Soil_type', 'Crop_type']

list

### Numerical, polynomial, & categorical transformers

**Conceptualy**

                         X_train
                            │
              ┌─────────────┴─────────────┐
              ↓                           ↓
        Numerical                    Categorical
              │                           │
              ↓                           ↓
    PolynomialFeatures              OneHotEncoder
              │                           │
              ↓                           │
       StandardScaler                     │
              │                           │
              └─────────────┬─────────────┘
                            ↓
                     LinearRegression

In [8]:
from sklearn.pipeline import Pipeline

def cat_num_transformers() -> Pipeline:
    numerical_transformer = Pipeline(
        steps=[
            ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler())
        ]
    )
    categorical_transformer = OneHotEncoder(
        handle_unknown="ignore"
    )
    return numerical_transformer, categorical_transformer

In [9]:
def combined_processor()-> ColumnTransformer:
    numerical_transformer, categorical_transformer = cat_num_transformers()
    polynomial_preprocessor = ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, num_features),
            ("cat", categorical_transformer, cat_features)
        ]
    )
    return polynomial_preprocessor


### Training the linear model

In [10]:
def polynomial_pipeline()-> Pipeline:
    polynomial_preprocessor = combined_processor()
    polynomial_model = Pipeline(
        steps=[
            ("preprocessor", polynomial_preprocessor),
            ("model", LinearRegression())
        ]
    )
    return polynomial_model

In [11]:
def polynomial_pipeline_model(X_train, X_test, y_train, y_test)-> Pipeline | np.ndarray:
    polynomial_model = polynomial_pipeline()
    polynomial_model.fit(X_train, y_train)
    y_pred_polynomial = polynomial_model.predict(X_test)
    return  polynomial_model, y_pred_polynomial

polynomial_model, y_pred_polynomial = polynomial_pipeline_model(X_train, X_test, y_train, y_test)
display(polynomial_model)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Elevation','Latitude','Longitude',...,'Pollution_level','Plot_size', 'Crop_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subse

## Model Evaluation

In [12]:
def polynomial_evaluation(y_test, y_pred_polynomial) ->tuple[float, float]:
    poly_r2_score = r2_score(y_test, y_pred_polynomial)
    poly_rmse = np.sqrt(mean_squared_error(y_test, y_pred_polynomial))
    return poly_r2_score, poly_rmse

poly_r2_score, poly_rmse = polynomial_evaluation(y_test, y_pred_polynomial)

print(f"polynomial r2 score: {poly_r2_score:.4}")
print(f"polynomial rmse: {poly_rmse:.4f}")



polynomial r2 score: 0.7389
polynomial rmse: 0.0590


## 4. Initial Polynomial Regression Results

A degree-2 polynomial regression model was evaluated to determine whether
nonlinear relationships and pairwise interactions could improve predictive
performance relative to the linear baseline.

The model achieved:

- RMSE: **0.0590**
- R²: **0.7389**

Compared with the OLS baseline:

- OLS RMSE: 0.0678
- Polynomial RMSE: 0.0590
- OLS R²: 0.6553
- Polynomial R²: 0.7389

The polynomial model produced a substantial improvement in both evaluation
metrics, supporting the hypothesis that nonlinear relationships and/or
interactions exist between the predictors and `Standard_yield`.

However, this improvement came with increased model complexity. The
transformed feature space increased from 31 features in the linear
representation to 109 features after polynomial expansion and categorical
encoding.

Therefore, the degree-2 polynomial model becomes the current best-performing
candidate, but additional validation is required before selecting it as the
final model.

### Cross-validation
**Does this model perform consistently across different subsets of the training data?**

In [13]:
from sklearn.model_selection import cross_val_score
def cross_validation(X_train, y_train):
    polynomial_model = polynomial_pipeline()
    cv_scores = cross_val_score(
        polynomial_model,
        X_train,
        y_train,
        cv = 5,
        scoring= "neg_root_mean_squared_error"
    )
    return  - cv_scores

cv_scores =  cross_validation(X_train, y_train)
cv_rmse_scores = cv_scores

print("CV RMSE scores:")
print(cv_rmse_scores)

print(f"\nMean cv RMSE: {cv_rmse_scores.mean():.4f}")
print(f"Standard deviation: {cv_rmse_scores.std():.4f}")

CV RMSE scores:
[0.06003643 0.05886734 0.05835946 0.05584774 0.05944676]

Mean cv RMSE: 0.0585
Standard deviation: 0.0014


### Cross-Validation Results

The degree-2 polynomial regression pipeline was evaluated using 5-fold
cross-validation on the training data. The entire preprocessing and modeling
pipeline was included in the cross-validation process to ensure that feature
transformation was fitted independently within each training fold.

The cross-validation RMSE scores were:

- Fold 1: 0.0600
- Fold 2: 0.0589
- Fold 3: 0.0584
- Fold 4: 0.0558
- Fold 5: 0.0594

The mean cross-validation RMSE was **0.0585** with a standard deviation of
**0.0014**.

The held-out test RMSE of **0.0590** is close to the mean cross-validation
RMSE, providing evidence that the model's performance is relatively stable
across different subsets of the data.

Therefore, the improvement observed relative to the linear baseline appears
to generalize consistently within the available dataset.

In [14]:
def evaluate_polynomial_degrees(degrees, X_train, y_train, num_features, cat_features, cv =5) -> pd.DataFrame:
    results =[]
    for degree in degrees:
        numerical_transformer = Pipeline(
            steps=[
                ("polynomial", PolynomialFeatures(degree=degree, include_bias=False)),
                ("scaler", StandardScaler())
                ]
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numerical_transformer, num_features),
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
            ]
        )
        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", LinearRegression())
            ]
        )
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
        rmse_scores = -scores
        results.append({
            "degree": degree,
            "mean_cv_rmse": rmse_scores.mean(),
            "std_cv_scores": rmse_scores.std()
        })

    return pd.DataFrame(results)

In [15]:
degrees = [1,2,3]

polynomial_results = evaluate_polynomial_degrees(
    degrees, X_train, y_train, num_features, cat_features
)

display(polynomial_results)

,degree,mean_cv_rmse,std_cv_scores
0,1,0.068234,0.001133
1,2,0.058512,0.001445
2,3,0.058798,0.001419


### Polynomial Degree Selection

Polynomial degrees 1, 2, and 3 were evaluated using 5-fold
cross-validation on the training data.

Degree 2 achieved the lowest mean cross-validation RMSE of 0.058512.

Moving from degree 1 to degree 2 resulted in a substantial improvement
in predictive performance, supporting the hypothesis that nonlinear
relationships and interactions are useful for predicting `Standard_yield`.

However, increasing the degree from 2 to 3 slightly increased the
cross-validation RMSE. This indicates that the additional complexity
introduced by degree 3 did not provide a predictive benefit.

Therefore, degree 2 is selected as the preferred polynomial complexity
for further evaluation.

## Polynomial Ridge Regression
**Can Ridge regularization control the increased complexity of the 109-feature polynomial representation and improve its generalization?**

In [16]:
from sklearn.linear_model import Ridge

In [17]:
def evaluate_polynomial_ridge(alphas, X_train, y_train, num_features, 
                              cat_features, degree = 2, cv =5)-> pd.DataFrame:
    results =[]
    for alpha in alphas:
        numerical_transformers = Pipeline(
            steps = [
                ("polynomial", PolynomialFeatures(degree = degree, include_bias=False)),
                ("scaler", StandardScaler())
            ]
        )
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numerical_transformers, num_features),
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
            ]
        )
        pipeline = Pipeline(
            steps = [
                ("preprpcessing", preprocessor),
                ("model", Ridge(alpha=alpha))
            ]
        )
        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring = "neg_root_mean_squared_error"
        )

        rmse_scores = -scores
        results.append({
            "alpha": alpha,
            "mean_cv_rmse": np.round(rmse_scores.mean(),7),
            "std_cv_rmse": np.round(rmse_scores.std(), 7)
        })
    return pd.DataFrame(results)

alphas = [0.001, 0.01, 0.1, 1, 10, 100]

polynomial_ridge_results = evaluate_polynomial_ridge(alphas, X_train, y_train, num_features, cat_features)
polynomial_ridge_results

,alpha,mean_cv_rmse,std_cv_rmse
0,0.001,0.058479,0.001442
1,0.010,0.058440,0.001443
2,0.100,0.058371,0.001439
3,1.000,0.058343,0.001413
4,10.000,0.059194,0.001395
5,100.000,0.064528,0.001293


## 7. Polynomial Ridge Regression

The degree-2 polynomial representation increased the feature space to 109
transformed features. Ridge regularization was therefore evaluated to
determine whether coefficient shrinkage could improve the generalization
of the polynomial model.

Six values of $\alpha$ were evaluated using 5-fold cross-validation.

| Alpha | Mean CV RMSE | Std. Dev. |
|------:|-------------:|----------:|
| 0.001 | 0.05848 | 0.00144 |
| 0.01 | 0.05844 | 0.00144 |
| 0.1 | 0.05837 | 0.00144 |
| 1.0 | **0.05834** | 0.00141 |
| 10.0 | 0.05919 | 0.00140 |
| 100.0 | 0.06453 | 0.00129 |

The best cross-validation performance was obtained with $\alpha=1$,
achieving a mean RMSE of 0.05834.

This is a small improvement over the unregularized degree-2 polynomial
model, which achieved a mean CV RMSE of 0.05851.

Increasing $\alpha$ beyond 1 resulted in progressively worse predictive
performance, suggesting that stronger regularization begins to remove
useful signal from the model.

Therefore, $\alpha=1$ is selected for further evaluation.

### Fit the Model

In [18]:
def polynomial_ridge_pipeline()-> Pipeline:
    numerical_transformer = Pipeline(
        steps=[
            ("polynomial", PolynomialFeatures(degree = 2,include_bias=False)),
            ("scaler", StandardScaler())
        ]
    )
    preprocessor = ColumnTransformer(
        transformers= [
                ("num", numerical_transformer, num_features),
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
        ]
    )
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", Ridge(alpha=1))
        ]
    )
    return pipeline


In [19]:
def polynomial_ridge_model(X_train, X_test, y_train)->Pipeline | np.ndarray:
    polynomial_pipeline = polynomial_ridge_pipeline()
    poly_ridge_model = polynomial_pipeline.fit(X_train, y_train)
    y_poly_ridge_pred = poly_ridge_model.predict(X_test)
    return poly_ridge_model, y_poly_ridge_pred

poly_ridge_model, y_poly_ridge_pred = polynomial_ridge_model(X_train, X_test, y_train)
display(poly_ridge_model)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Elevation','Latitude','Longitude',...,'Pollution_level','Plot_size', 'Crop_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subse

In [20]:
def poly_ridge_evaluatin(y_test, y_poly_ridge_pred) -> tuple[float, float]:
    polynomial_ridge_rmse = np.sqrt(mean_squared_error(y_test, y_poly_ridge_pred))
    polynomial_ridge_r2 = r2_score(y_test, y_poly_ridge_pred)
    return polynomial_ridge_rmse, polynomial_ridge_r2

polynomial_ridge_rmse, polynomial_ridge_r2 = poly_ridge_evaluatin(y_test, y_poly_ridge_pred)

print(f"rmse: {polynomial_ridge_rmse:.4f}")
print(f"r2 score{polynomial_ridge_r2:.4f}")

rmse: 0.0590
r2 score0.7391


## 8. Evaluation of Polynomial Ridge

The polynomial Ridge model was configured using the hyperparameters
selected during cross-validation:

- Polynomial degree: 2
- Ridge alpha: 1

The model was fitted on the full training set and evaluated on the
held-out test set.

### Test Results

- RMSE: 0.0590
- R²: 0.7391

The Polynomial Ridge model achieved a small improvement in R² compared
with the unregularized degree-2 polynomial model. However, the
difference in predictive performance was minimal.

The results suggest that the primary performance improvement came from
capturing nonlinear relationships and interactions through polynomial
features, while Ridge regularization provided only a marginal additional
benefit.

Therefore, Polynomial Ridge is currently the strongest candidate model,
although its advantage over the unregularized polynomial model is small.

### Lasso Regression

**OBJECTIVE**

To evaluate polynomial features with lassoCV regresion  to determine whether to model nonlinear relatioships improve the predictive performance of the model compared with previous regressions models. 

In [21]:
from sklearn.linear_model import LassoCV

def build_polynomial_lassocv_pipeline()-> Pipeline:
    num_preprocessor = Pipeline(
        steps=[
            ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler())
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_preprocessor, num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
        ]
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LassoCV(max_iter=10000))
        ]
    )
    return pipeline

In [22]:
def fit_polynomial_lassocv_pipelie(X_train, y_train, X_test) -> Pipeline | np.ndarray:
    pipeline = build_polynomial_lassocv_pipeline()
    polynomial_lassocv = pipeline.fit(X_train, y_train)
    y_pred_poly_lassocv = polynomial_lassocv.predict(X_test)
    return polynomial_lassocv, y_pred_poly_lassocv

polynomial_lassocv, y_pred_poly_lassocv = fit_polynomial_lassocv_pipelie(X_train, y_train, X_test)
display(polynomial_lassocv)

/home/dataflix/Projects/maji-ndogo-yield-intelligence/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.920184e-02, tolerance: 5.555e-03
  model = cd_fast.enet_coordinate_descent(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Elevation','Latitude','Longitude',...,'Pollution_level','Plot_size', 'Crop_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subse

In [23]:
def poly_lossocv_evaluation(y_test, y_pred_poly_lassocv) -> tuple[float, float]:
    polycv_rmse = np.sqrt(mean_squared_error(y_test, y_pred_poly_lassocv))
    polycv_r2_score= r2_score(y_test, y_pred_poly_lassocv)
    return polycv_rmse, polycv_r2_score

polycv_rmse, polycv_r2_score = poly_lossocv_evaluation(y_test, y_pred_poly_lassocv)

print(f"rmse: {polycv_rmse:.4f}")
print(f"r2 score: {polycv_r2_score:.4f}")

rmse: 0.0589
r2 score: 0.7396


## Conclusions

The polynomial features plus lasssoCV regression model captured nonlinear patterns in the data by generating additional features which enabled the model to learn more complex relationships between the predictors and the target variable. The  model achieved a lower rmse and higher r2 than previous linear regression models indicating an improvement in predictive performance. These results suggest that incoperating nonlinear features improved the model's ability to capture non linear relationships resulting in more accurate predictions of standard_yield.